# Database MCP Server — Model Context Protocol

##Project Overview

This project implements a production-style Model Context Protocol (MCP) server that exposes a structured database through MCP tools.

The server demonstrates how an AI application can interact with external data and perform controlled database operations through the Model Context Protocol, instead of directly accessing the database.

## Key Components
MCP Server — exposes database capabilities as MCP tools
SQLite Database — lightweight persistent data storage
MCP Client Demo — demonstrates programmatic interaction with the server
Automated Tests — validates server and database functionality
MCP Inspector — provides an interactive interface for inspecting and testing MCP tools
Seed Script — initializes the database with sample data

In [1]:
!pip install -q "mcp[cli]" pytest

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.7/69.7 kB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 357.3/357.3 kB 5.6 MB/s eta 0:00:00


In [2]:
!mkdir -p /content/database-mcp-server/tests
%cd /content/database-mcp-server

/content/database-mcp-server


In [3]:
%%writefile database.py

"""
database.py

Reusable SQLite database module for the Database MCP Server.

Responsibilities:
    * database file path
    * connection creation (with foreign keys enabled)
    * schema creation
    * small, safe, parameterized query helpers

This module has NO knowledge of MCP. The MCP server layer (server.py)
calls into these functions. Keeping the two layers separate makes the
database logic independently testable and reusable.

All SQL in this module uses parameterized queries (`?` placeholders).
User-supplied values are NEVER concatenated into SQL strings.
"""

from __future__ import annotations

import sqlite3
from contextlib import contextmanager
from pathlib import Path
from typing import Any, Iterator

# ---------------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------------

# Default database file lives next to this module.
DB_PATH = Path(__file__).resolve().parent / "ecommerce.db"

# Business rules enforced at the application layer (and mirrored with
# SQL CHECK constraints at the schema layer).
ALLOWED_SEGMENTS = ("Standard", "Premium", "Enterprise")
ALLOWED_STATUSES = ("pending", "processing", "shipped", "delivered", "cancelled")


# ---------------------------------------------------------------------------
# Connections
# ---------------------------------------------------------------------------

def get_connection(db_path: str | Path = DB_PATH) -> sqlite3.Connection:
    """Create a new SQLite connection with foreign keys enabled.

    Each call returns a fresh connection. Callers are responsible for
    closing it (the `connection()` context manager below does this
    automatically).
    """
    conn = sqlite3.connect(str(db_path))
    conn.row_factory = sqlite3.Row
    conn.execute("PRAGMA foreign_keys = ON;")
    return conn


@contextmanager
def connection(db_path: str | Path = DB_PATH) -> Iterator[sqlite3.Connection]:
    """Context manager that yields a connection and always closes it.

    Commits on success, rolls back on exception.
    """
    conn = get_connection(db_path)
    try:
        yield conn
        conn.commit()
    except Exception:
        conn.rollback()
        raise
    finally:
        conn.close()


# ---------------------------------------------------------------------------
# Schema
# ---------------------------------------------------------------------------

SCHEMA_SQL = """
CREATE TABLE IF NOT EXISTS customers (
    id          INTEGER PRIMARY KEY AUTOINCREMENT,
    name        TEXT    NOT NULL,
    email       TEXT    NOT NULL UNIQUE,
    country     TEXT    NOT NULL,
    segment     TEXT    NOT NULL CHECK (segment IN ('Standard', 'Premium', 'Enterprise')),
    created_at  TEXT    NOT NULL
);

CREATE TABLE IF NOT EXISTS products (
    id          INTEGER PRIMARY KEY AUTOINCREMENT,
    name        TEXT    NOT NULL,
    category    TEXT    NOT NULL,
    price       REAL    NOT NULL CHECK (price >= 0),
    stock       INTEGER NOT NULL CHECK (stock >= 0)
);

CREATE TABLE IF NOT EXISTS orders (
    id            INTEGER PRIMARY KEY AUTOINCREMENT,
    customer_id   INTEGER NOT NULL,
    order_date    TEXT    NOT NULL,
    status        TEXT    NOT NULL CHECK (
                      status IN ('pending', 'processing', 'shipped', 'delivered', 'cancelled')
                  ),
    total_amount  REAL    NOT NULL CHECK (total_amount >= 0),
    FOREIGN KEY (customer_id) REFERENCES customers (id)
);

CREATE TABLE IF NOT EXISTS order_items (
    id          INTEGER PRIMARY KEY AUTOINCREMENT,
    order_id    INTEGER NOT NULL,
    product_id  INTEGER NOT NULL,
    quantity    INTEGER NOT NULL CHECK (quantity > 0),
    unit_price  REAL    NOT NULL CHECK (unit_price >= 0),
    FOREIGN KEY (order_id)   REFERENCES orders (id),
    FOREIGN KEY (product_id) REFERENCES products (id)
);

CREATE INDEX IF NOT EXISTS idx_orders_customer_id ON orders (customer_id);
CREATE INDEX IF NOT EXISTS idx_order_items_order_id ON order_items (order_id);
CREATE INDEX IF NOT EXISTS idx_order_items_product_id ON order_items (product_id);
"""


def init_schema(db_path: str | Path = DB_PATH) -> None:
    """Create all tables (and indexes) if they do not already exist."""
    with connection(db_path) as conn:
        conn.executescript(SCHEMA_SQL)


def reset_schema(db_path: str | Path = DB_PATH) -> None:
    """Drop and recreate all tables. Used to make seeding idempotent."""
    with connection(db_path) as conn:
        conn.executescript(
            """
            DROP TABLE IF EXISTS order_items;
            DROP TABLE IF EXISTS orders;
            DROP TABLE IF EXISTS products;
            DROP TABLE IF EXISTS customers;
            """
        )
        conn.executescript(SCHEMA_SQL)


# ---------------------------------------------------------------------------
# Generic helpers
# ---------------------------------------------------------------------------

def rows_to_dicts(rows: list[sqlite3.Row]) -> list[dict[str, Any]]:
    """Convert a list of sqlite3.Row objects into plain dicts."""
    return [dict(row) for row in rows]


def table_counts(db_path: str | Path = DB_PATH) -> dict[str, int]:
    """Return row counts for every table. Useful for seeding summaries/tests."""
    counts: dict[str, int] = {}
    with connection(db_path) as conn:
        for table in ("customers", "products", "orders", "order_items"):
            cur = conn.execute(f"SELECT COUNT(*) AS n FROM {table}")
            counts[table] = cur.fetchone()["n"]
    return counts


# ---------------------------------------------------------------------------
# Insert helpers (used by seed_database.py)
# ---------------------------------------------------------------------------

def insert_customer(
    conn: sqlite3.Connection,
    name: str,
    email: str,
    country: str,
    segment: str,
    created_at: str,
) -> int:
    """Insert a customer row using a parameterized query. Returns new id."""
    if segment not in ALLOWED_SEGMENTS:
        raise ValueError(f"Invalid segment: {segment!r}")
    cur = conn.execute(
        """
        INSERT INTO customers (name, email, country, segment, created_at)
        VALUES (?, ?, ?, ?, ?)
        """,
        (name, email, country, segment, created_at),
    )
    return int(cur.lastrowid)


def insert_product(
    conn: sqlite3.Connection,
    name: str,
    category: str,
    price: float,
    stock: int,
) -> int:
    """Insert a product row using a parameterized query. Returns new id."""
    cur = conn.execute(
        "INSERT INTO products (name, category, price, stock) VALUES (?, ?, ?, ?)",
        (name, category, price, stock),
    )
    return int(cur.lastrowid)


def insert_order(
    conn: sqlite3.Connection,
    customer_id: int,
    order_date: str,
    status: str,
    total_amount: float,
) -> int:
    """Insert an order row using a parameterized query. Returns new id."""
    if status not in ALLOWED_STATUSES:
        raise ValueError(f"Invalid status: {status!r}")
    cur = conn.execute(
        """
        INSERT INTO orders (customer_id, order_date, status, total_amount)
        VALUES (?, ?, ?, ?)
        """,
        (customer_id, order_date, status, total_amount),
    )
    return int(cur.lastrowid)


def insert_order_item(
    conn: sqlite3.Connection,
    order_id: int,
    product_id: int,
    quantity: int,
    unit_price: float,
) -> int:
    """Insert an order_item row using a parameterized query. Returns new id."""
    cur = conn.execute(
        """
        INSERT INTO order_items (order_id, product_id, quantity, unit_price)
        VALUES (?, ?, ?, ?)
        """,
        (order_id, product_id, quantity, unit_price),
    )
    return int(cur.lastrowid)


# ---------------------------------------------------------------------------
# Read helpers (used by server.py tools/resources)
# ---------------------------------------------------------------------------

def query_customers(
    db_path: str | Path,
    name: str | None = None,
    country: str | None = None,
    segment: str | None = None,
    limit: int = 10,
) -> list[dict[str, Any]]:
    """Search customers with optional, safely-parameterized filters."""
    clauses: list[str] = []
    params: list[Any] = []

    if name:
        clauses.append("name LIKE ?")
        params.append(f"%{name}%")
    if country:
        clauses.append("country = ?")
        params.append(country)
    if segment:
        clauses.append("segment = ?")
        params.append(segment)

    where_sql = f"WHERE {' AND '.join(clauses)}" if clauses else ""
    sql = f"""
        SELECT id, name, email, country, segment, created_at
        FROM customers
        {where_sql}
        ORDER BY id
        LIMIT ?
    """
    params.append(limit)

    with connection(db_path) as conn:
        cur = conn.execute(sql, params)
        return rows_to_dicts(cur.fetchall())


def get_customer_by_id(db_path: str | Path, customer_id: int) -> dict[str, Any] | None:
    """Fetch a single customer row by id, or None if not found."""
    with connection(db_path) as conn:
        cur = conn.execute(
            "SELECT id, name, email, country, segment, created_at "
            "FROM customers WHERE id = ?",
            (customer_id,),
        )
        row = cur.fetchone()
        return dict(row) if row else None


def customer_order_summary(db_path: str | Path, customer_id: int) -> dict[str, Any] | None:
    """Return order analytics for a single customer, or None if the
    customer does not exist."""
    customer = get_customer_by_id(db_path, customer_id)
    if customer is None:
        return None

    with connection(db_path) as conn:
        agg = conn.execute(
            """
            SELECT
                COUNT(*)                AS order_count,
                COALESCE(SUM(total_amount), 0.0)  AS total_spent,
                COALESCE(AVG(total_amount), 0.0)  AS average_order_value,
                MAX(order_date)         AS last_order_date
            FROM orders
            WHERE customer_id = ?
            """,
            (customer_id,),
        ).fetchone()

        status_rows = conn.execute(
            """
            SELECT status, COUNT(*) AS n
            FROM orders
            WHERE customer_id = ?
            GROUP BY status
            """,
            (customer_id,),
        ).fetchall()

    status_breakdown = {row["status"]: row["n"] for row in status_rows}

    return {
        "customer": customer,
        "order_count": agg["order_count"],
        "total_spent": round(agg["total_spent"], 2),
        "average_order_value": round(agg["average_order_value"], 2),
        "last_order_date": agg["last_order_date"],
        "status_breakdown": status_breakdown,
    }


def top_products(
    db_path: str | Path,
    limit: int = 5,
    category: str | None = None,
) -> list[dict[str, Any]]:
    """Return the highest-revenue products, optionally filtered by category."""
    clauses: list[str] = []
    params: list[Any] = []

    if category:
        clauses.append("p.category = ?")
        params.append(category)

    where_sql = f"WHERE {' AND '.join(clauses)}" if clauses else ""
    sql = f"""
        SELECT
            p.id                                   AS product_id,
            p.name                                  AS product_name,
            p.category                              AS category,
            COALESCE(SUM(oi.quantity), 0)           AS units_sold,
            COALESCE(SUM(oi.quantity * oi.unit_price), 0.0) AS revenue
        FROM products p
        LEFT JOIN order_items oi ON oi.product_id = p.id
        LEFT JOIN orders o ON o.id = oi.order_id AND o.status != 'cancelled'
        {where_sql}
        GROUP BY p.id
        ORDER BY revenue DESC, units_sold DESC
        LIMIT ?
    """
    params.append(limit)

    with connection(db_path) as conn:
        cur = conn.execute(sql, params)
        rows = rows_to_dicts(cur.fetchall())

    for row in rows:
        row["revenue"] = round(row["revenue"], 2)
    return rows


def sales_summary(
    db_path: str | Path,
    start_date: str | None = None,
    end_date: str | None = None,
) -> dict[str, Any]:
    """Return overall business analytics, optionally scoped to a date range."""
    clauses: list[str] = []
    params: list[Any] = []

    if start_date:
        clauses.append("order_date >= ?")
        params.append(start_date)
    if end_date:
        clauses.append("order_date <= ?")
        params.append(end_date)

    where_sql = f"WHERE {' AND '.join(clauses)}" if clauses else ""

    with connection(db_path) as conn:
        agg = conn.execute(
            f"""
            SELECT
                COUNT(*)                                            AS total_orders,
                COALESCE(SUM(total_amount), 0.0)                    AS total_revenue,
                COALESCE(AVG(total_amount), 0.0)                    AS average_order_value,
                COUNT(DISTINCT customer_id)                         AS unique_customers,
                SUM(CASE WHEN status = 'cancelled' THEN 1 ELSE 0 END) AS cancelled_orders
            FROM orders
            {where_sql}
            """,
            params,
        ).fetchone()

        top_category_row = conn.execute(
            f"""
            SELECT p.category AS category, SUM(oi.quantity * oi.unit_price) AS revenue
            FROM order_items oi
            JOIN orders o ON o.id = oi.order_id
            JOIN products p ON p.id = oi.product_id
            {where_sql.replace("order_date", "o.order_date") if where_sql else ""}
            GROUP BY p.category
            ORDER BY revenue DESC
            LIMIT 1
            """,
            params,
        ).fetchone()

    return {
        "total_orders": agg["total_orders"],
        "total_revenue": round(agg["total_revenue"], 2),
        "average_order_value": round(agg["average_order_value"], 2),
        "unique_customers": agg["unique_customers"],
        "cancelled_orders": agg["cancelled_orders"] or 0,
        "top_category": top_category_row["category"] if top_category_row else None,
    }


def order_status_summary(db_path: str | Path) -> dict[str, int]:
    """Return a count of orders grouped by status.

    All known statuses are always present in the result (defaulting to 0)
    so downstream consumers get a predictable, complete shape.
    """
    counts = {status: 0 for status in ALLOWED_STATUSES}
    with connection(db_path) as conn:
        rows = conn.execute(
            "SELECT status, COUNT(*) AS n FROM orders GROUP BY status"
        ).fetchall()
    for row in rows:
        counts[row["status"]] = row["n"]
    return counts


def customer_context(db_path: str | Path, customer_id: int) -> dict[str, Any] | None:
    """Return a readable context bundle for a single customer: profile,
    order stats, and recent orders. Used by the dynamic MCP resource."""
    customer = get_customer_by_id(db_path, customer_id)
    if customer is None:
        return None

    with connection(db_path) as conn:
        agg = conn.execute(
            """
            SELECT COUNT(*) AS order_count, COALESCE(SUM(total_amount), 0.0) AS total_spent
            FROM orders WHERE customer_id = ?
            """,
            (customer_id,),
        ).fetchone()

        recent_orders = conn.execute(
            """
            SELECT id, order_date, status, total_amount
            FROM orders
            WHERE customer_id = ?
            ORDER BY order_date DESC
            LIMIT 5
            """,
            (customer_id,),
        ).fetchall()

    return {
        "customer": customer,
        "order_count": agg["order_count"],
        "total_spent": round(agg["total_spent"], 2),
        "recent_orders": rows_to_dicts(recent_orders),
    }


Writing database.py


In [4]:
%%writefile seed_database.py

"""
seed_database.py

Creates the SQLite database (if needed), (re)creates the schema, and
inserts a deterministic set of sample e-commerce data:

    * 20 customers  (multiple countries, multiple segments)
    * 10 products   (multiple categories)
    * 40 orders     (multiple statuses, spread across dates)
    * order_items for every order

Running this script is always safe: it resets the schema first, so
re-running it never creates duplicate data.

Usage:
    python seed_database.py
"""

from __future__ import annotations

import database as db


# ---------------------------------------------------------------------------
# Deterministic seed data
# ---------------------------------------------------------------------------
# Everything below is hard-coded (not randomly generated) so that the
# database — and therefore the tests and the demo output — is identical
# on every run.

CUSTOMERS = [
    # (name, email, country, segment, created_at)
    ("Alice Johnson", "alice.johnson@example.com", "USA", "Premium", "2023-01-05"),
    ("Bruno Silva", "bruno.silva@example.com", "Brazil", "Standard", "2023-01-12"),
    ("Chen Wei", "chen.wei@example.com", "China", "Enterprise", "2023-01-20"),
    ("Diana Kovacs", "diana.kovacs@example.com", "Hungary", "Standard", "2023-02-02"),
    ("Ethan Brown", "ethan.brown@example.com", "USA", "Standard", "2023-02-10"),
    ("Fatima Al-Sayed", "fatima.alsayed@example.com", "UAE", "Premium", "2023-02-18"),
    ("George Papadopoulos", "george.p@example.com", "Greece", "Standard", "2023-03-01"),
    ("Hannah Schmidt", "hannah.schmidt@example.com", "Germany", "Premium", "2023-03-09"),
    ("Ivan Petrov", "ivan.petrov@example.com", "Russia", "Standard", "2023-03-15"),
    ("Julia Nowak", "julia.nowak@example.com", "Poland", "Standard", "2023-03-22"),
    ("Kenji Yamamoto", "kenji.yamamoto@example.com", "Japan", "Enterprise", "2023-04-01"),
    ("Laura Martinez", "laura.martinez@example.com", "Mexico", "Standard", "2023-04-08"),
    ("Mohammed Khan", "mohammed.khan@example.com", "Pakistan", "Standard", "2023-04-16"),
    ("Nora Andersen", "nora.andersen@example.com", "Denmark", "Premium", "2023-04-25"),
    ("Oliver Smith", "oliver.smith@example.com", "UK", "Standard", "2023-05-03"),
    ("Priya Sharma", "priya.sharma@example.com", "India", "Enterprise", "2023-05-11"),
    ("Quinn O'Brien", "quinn.obrien@example.com", "Ireland", "Standard", "2023-05-19"),
    ("Rosa Fernandez", "rosa.fernandez@example.com", "Spain", "Premium", "2023-05-27"),
    ("Samuel Osei", "samuel.osei@example.com", "Ghana", "Standard", "2023-06-04"),
    ("Tina Nguyen", "tina.nguyen@example.com", "Vietnam", "Standard", "2023-06-12"),
]

PRODUCTS = [
    # (name, category, price, stock)
    ("Wireless Mouse", "Electronics", 24.99, 150),
    ("Mechanical Keyboard", "Electronics", 79.99, 90),
    ("USB-C Hub", "Electronics", 34.50, 120),
    ("Noise Cancelling Headphones", "Electronics", 159.99, 60),
    ("Standing Desk", "Furniture", 349.00, 25),
    ("Ergonomic Office Chair", "Furniture", 219.00, 40),
    ("Stainless Steel Water Bottle", "Home & Kitchen", 18.75, 200),
    ("Ceramic Coffee Mug Set", "Home & Kitchen", 22.00, 180),
    ("Running Shoes", "Apparel", 89.99, 75),
    ("Fleece Jacket", "Apparel", 64.50, 100),
]

# (customer_index[1-based], order_date, status, item list)
# item list entries: (product_index[1-based], quantity)
ORDERS: list[tuple[int, str, str, list[tuple[int, int]]]] = [
    (1, "2023-06-01", "delivered", [(1, 2), (3, 1)]),
    (2, "2023-06-02", "delivered", [(7, 3)]),
    (3, "2023-06-03", "delivered", [(5, 1), (6, 1)]),
    (4, "2023-06-04", "cancelled", [(9, 1)]),
    (5, "2023-06-05", "delivered", [(2, 1)]),
    (6, "2023-06-06", "shipped", [(4, 1), (1, 1)]),
    (7, "2023-06-07", "delivered", [(8, 2)]),
    (8, "2023-06-08", "delivered", [(6, 1), (5, 1)]),
    (9, "2023-06-09", "pending", [(10, 1)]),
    (10, "2023-06-10", "delivered", [(7, 1), (8, 1)]),
    (1, "2023-06-12", "delivered", [(4, 1)]),
    (11, "2023-06-13", "delivered", [(5, 2), (6, 2)]),
    (12, "2023-06-14", "processing", [(9, 1), (10, 1)]),
    (13, "2023-06-15", "delivered", [(2, 1), (3, 1)]),
    (14, "2023-06-16", "delivered", [(4, 1)]),
    (15, "2023-06-17", "shipped", [(1, 3)]),
    (16, "2023-06-18", "delivered", [(6, 1)]),
    (17, "2023-06-19", "delivered", [(7, 2), (8, 2)]),
    (18, "2023-06-20", "delivered", [(4, 1), (2, 1)]),
    (19, "2023-06-21", "cancelled", [(9, 1)]),
    (20, "2023-06-22", "delivered", [(10, 1), (9, 1)]),
    (2, "2023-06-24", "delivered", [(3, 2)]),
    (3, "2023-06-25", "delivered", [(5, 1)]),
    (6, "2023-06-26", "pending", [(1, 1), (2, 1)]),
    (8, "2023-06-27", "delivered", [(4, 2)]),
    (11, "2023-06-28", "delivered", [(6, 1), (7, 1)]),
    (14, "2023-06-29", "shipped", [(8, 1)]),
    (16, "2023-06-30", "delivered", [(9, 2)]),
    (1, "2023-07-01", "delivered", [(10, 1)]),
    (5, "2023-07-02", "delivered", [(1, 1), (4, 1)]),
    (7, "2023-07-03", "processing", [(2, 2)]),
    (9, "2023-07-04", "delivered", [(3, 1), (5, 1)]),
    (10, "2023-07-05", "delivered", [(6, 2)]),
    (12, "2023-07-06", "delivered", [(7, 1)]),
    (13, "2023-07-07", "cancelled", [(8, 1)]),
    (15, "2023-07-08", "delivered", [(9, 1), (10, 1)]),
    (17, "2023-07-09", "delivered", [(1, 2)]),
    (18, "2023-07-10", "pending", [(2, 1)]),
    (19, "2023-07-11", "delivered", [(3, 1), (4, 1)]),
    (20, "2023-07-12", "delivered", [(5, 1), (6, 1)]),
]


def seed(db_path=db.DB_PATH) -> dict[str, int]:
    """Reset the schema and insert deterministic sample data.

    Returns the resulting table row counts.
    """
    db.reset_schema(db_path)

    with db.connection(db_path) as conn:
        customer_ids: list[int] = []
        for name, email, country, segment, created_at in CUSTOMERS:
            cid = db.insert_customer(conn, name, email, country, segment, created_at)
            customer_ids.append(cid)

        product_ids: list[int] = []
        for name, category, price, stock in PRODUCTS:
            pid = db.insert_product(conn, name, category, price, stock)
            product_ids.append(pid)

        product_price_by_id = dict(zip(product_ids, (p[2] for p in PRODUCTS)))

        for customer_idx, order_date, status, items in ORDERS:
            customer_id = customer_ids[customer_idx - 1]

            total_amount = 0.0
            for product_idx, quantity in items:
                unit_price = PRODUCTS[product_idx - 1][2]
                total_amount += unit_price * quantity
            total_amount = round(total_amount, 2)

            order_id = db.insert_order(conn, customer_id, order_date, status, total_amount)

            for product_idx, quantity in items:
                product_id = product_ids[product_idx - 1]
                unit_price = product_price_by_id[product_id]
                db.insert_order_item(conn, order_id, product_id, quantity, unit_price)

    return db.table_counts(db_path)


def main() -> None:
    counts = seed()
    print("Database initialized successfully.\n")
    print(f"Customers:    {counts['customers']}")
    print(f"Products:     {counts['products']}")
    print(f"Orders:       {counts['orders']}")
    print(f"Order Items:  {counts['order_items']}")
    print(f"\nDatabase file: {db.DB_PATH}")


if __name__ == "__main__":
    main()


Writing seed_database.py


In [5]:
%%writefile server.py

"""
server.py

Database MCP Server.

Exposes a small e-commerce analytics SQLite database to an AI agent
through the Model Context Protocol (MCP):

    * Tools     -> safe, parameterized business operations
    * Resources -> read-only schema + customer context documents
    * Prompt    -> a structured analytics prompt template

SAFETY DESIGN
-------------
This server intentionally does NOT expose a generic "run SQL" tool.
There is no `execute_sql`, `run_query`, or `raw_sql` tool anywhere in
this file. Every tool below accepts a small set of typed, validated
parameters and internally builds a single parameterized SQL query
(see database.py). This means an AI agent driving this server can
never submit arbitrary SQL, regardless of how it is prompted.

    Agent -> Safe MCP Tool -> Validated Parameters -> Parameterized SQL -> SQLite

See the README ("Safety Architecture") for the full rationale.
"""

from __future__ import annotations

import re
from pathlib import Path
from typing import Any

import database as db
from mcp.server import MCPServer
from mcp.server.mcpserver.exceptions import ToolError

# ---------------------------------------------------------------------------
# Server instance
# ---------------------------------------------------------------------------

mcp = MCPServer(
    name="database-mcp-server",
    title="Database MCP Server",
    instructions=(
        "This server exposes a read-only view of a small e-commerce "
        "analytics database. Use the tools to search customers, analyze "
        "orders, and retrieve sales analytics. Read the 'db://schema' "
        "resource first to understand the data model. There is no tool "
        "for arbitrary SQL execution by design."
    ),
    version="1.0.0",
)

_DATE_RE = re.compile(r"^\d{4}-\d{2}-\d{2}$")


def _validate_date(value: str, field_name: str) -> None:
    """Validate an ISO date string (YYYY-MM-DD). Raises ToolError if invalid."""
    if not _DATE_RE.match(value):
        raise ToolError(
            f"Invalid {field_name}: {value!r}. Expected ISO format YYYY-MM-DD."
        )


def _validate_limit(limit: int, minimum: int = 1, maximum: int = 20) -> None:
    """Validate that a limit parameter is within an allowed range."""
    if not (minimum <= limit <= maximum):
        raise ToolError(
            f"Invalid limit: {limit}. Must be between {minimum} and {maximum}."
        )


# ---------------------------------------------------------------------------
# TOOL 1 — query_customers
# ---------------------------------------------------------------------------

@mcp.tool()
def query_customers(
    name: str | None = None,
    country: str | None = None,
    segment: str | None = None,
    limit: int = 10,
) -> dict[str, Any]:
    """Search customers using safe, optional filters.

    Args:
        name: Optional substring to match against the customer's name.
        country: Optional exact country match.
        segment: Optional exact segment match ('Standard', 'Premium', 'Enterprise').
        limit: Maximum number of results to return (1-20). Defaults to 10.

    Returns:
        A dict with the matching customers and the count returned.
    """
    _validate_limit(limit)

    if segment is not None and segment not in db.ALLOWED_SEGMENTS:
        raise ToolError(
            f"Invalid segment: {segment!r}. Allowed values: {list(db.ALLOWED_SEGMENTS)}."
        )

    try:
        customers = db.query_customers(
            db.DB_PATH, name=name, country=country, segment=segment, limit=limit
        )
    except Exception as exc:  # pragma: no cover - defensive, DB errors are rare
        raise ToolError(f"Database error while searching customers: {exc}") from exc

    return {"customers": customers, "count": len(customers)}


# ---------------------------------------------------------------------------
# TOOL 2 — customer_order_summary
# ---------------------------------------------------------------------------

@mcp.tool()
def customer_order_summary(customer_id: int) -> dict[str, Any]:
    """Return order analytics for a single customer.

    Args:
        customer_id: The numeric id of the customer.

    Returns:
        A dict containing the customer's profile, order_count,
        total_spent, average_order_value, last_order_date, and a
        status_breakdown (counts of orders per status).
    """
    if customer_id <= 0:
        raise ToolError(f"Invalid customer_id: {customer_id}. Must be a positive integer.")

    try:
        summary = db.customer_order_summary(db.DB_PATH, customer_id)
    except Exception as exc:  # pragma: no cover
        raise ToolError(f"Database error while summarizing customer orders: {exc}") from exc

    if summary is None:
        raise ToolError(f"Customer not found: {customer_id}")

    return summary


# ---------------------------------------------------------------------------
# TOOL 3 — top_products
# ---------------------------------------------------------------------------

@mcp.tool()
def top_products(limit: int = 5, category: str | None = None) -> dict[str, Any]:
    """Identify the highest-revenue products, optionally within a category.

    Args:
        limit: Maximum number of products to return (1-20). Defaults to 5.
        category: Optional exact category match.

    Returns:
        A dict with a list of products (product_id, product_name,
        category, units_sold, revenue), sorted by revenue descending.
    """
    _validate_limit(limit)

    try:
        products = db.top_products(db.DB_PATH, limit=limit, category=category)
    except Exception as exc:  # pragma: no cover
        raise ToolError(f"Database error while ranking products: {exc}") from exc

    return {"products": products, "count": len(products)}


# ---------------------------------------------------------------------------
# TOOL 4 — sales_summary
# ---------------------------------------------------------------------------

@mcp.tool()
def sales_summary(
    start_date: str | None = None,
    end_date: str | None = None,
) -> dict[str, Any]:
    """Provide overall business analytics, optionally scoped to a date range.

    Args:
        start_date: Optional inclusive ISO start date (YYYY-MM-DD).
        end_date: Optional inclusive ISO end date (YYYY-MM-DD).

    Returns:
        A dict with total_orders, total_revenue, average_order_value,
        unique_customers, cancelled_orders, and top_category.
    """
    if start_date is not None:
        _validate_date(start_date, "start_date")
    if end_date is not None:
        _validate_date(end_date, "end_date")
    if start_date is not None and end_date is not None and start_date > end_date:
        raise ToolError(
            f"Invalid date range: start_date ({start_date}) is after end_date ({end_date})."
        )

    try:
        return db.sales_summary(db.DB_PATH, start_date=start_date, end_date=end_date)
    except Exception as exc:  # pragma: no cover
        raise ToolError(f"Database error while computing sales summary: {exc}") from exc


# ---------------------------------------------------------------------------
# TOOL 5 — order_status_summary
# ---------------------------------------------------------------------------

@mcp.tool()
def order_status_summary() -> dict[str, int]:
    """Show the distribution of orders across all statuses.

    Returns:
        A dict mapping each order status to its count, e.g.
        {"pending": 4, "processing": 5, "shipped": 8, "delivered": 20, "cancelled": 3}.
    """
    try:
        return db.order_status_summary(db.DB_PATH)
    except Exception as exc:  # pragma: no cover
        raise ToolError(f"Database error while summarizing order statuses: {exc}") from exc


# ---------------------------------------------------------------------------
# RESOURCE 1 — db://schema (static)
# ---------------------------------------------------------------------------

_SCHEMA_TEXT = """\
DATABASE SCHEMA — E-Commerce Analytics
=======================================

customers
---------
  id          INTEGER  primary key
  name        TEXT     full name of the customer
  email       TEXT     unique contact email
  country     TEXT     customer's country
  segment     TEXT     one of: Standard, Premium, Enterprise
  created_at  TEXT     ISO date the customer record was created

products
--------
  id       INTEGER  primary key
  name     TEXT     product name
  category TEXT     product category (e.g. Electronics, Furniture)
  price    REAL     unit list price
  stock    INTEGER  units currently in stock

orders
------
  id            INTEGER  primary key
  customer_id   INTEGER  foreign key -> customers.id
  order_date    TEXT     ISO date (YYYY-MM-DD) the order was placed
  status        TEXT     one of: pending, processing, shipped, delivered, cancelled
  total_amount  REAL     total value of the order

order_items
-----------
  id          INTEGER  primary key
  order_id    INTEGER  foreign key -> orders.id
  product_id  INTEGER  foreign key -> products.id
  quantity    INTEGER  units of the product purchased in this order
  unit_price  REAL     price per unit at the time of purchase

Relationships
-------------
  customers (1) --- (many) orders
  orders    (1) --- (many) order_items
  products  (1) --- (many) order_items

Business notes
--------------
  * "revenue" for a product = SUM(order_items.quantity * order_items.unit_price)
    across non-cancelled orders.
  * A customer's "total_spent" = SUM(orders.total_amount) across all of
    their orders (all statuses, including cancelled, unless a tool says
    otherwise).
  * Use the provided tools to query this data. There is no tool for
    arbitrary SQL execution.
"""


@mcp.resource("db://schema")
def schema_resource() -> str:
    """Human-readable description of the database schema, relationships,
    and business meanings. Read this before running analytics."""
    return _SCHEMA_TEXT


# ---------------------------------------------------------------------------
# RESOURCE 2 — db://customer/{customer_id} (dynamic)
# ---------------------------------------------------------------------------

@mcp.resource("db://customer/{customer_id}")
def customer_resource(customer_id: str) -> str:
    """Readable context for a single customer: profile, order count,
    total spent, and recent orders. Handles nonexistent customers cleanly."""
    try:
        cid = int(customer_id)
    except ValueError:
        return f"Invalid customer id: {customer_id!r}. Expected an integer."

    context = db.customer_context(db.DB_PATH, cid)
    if context is None:
        return f"No customer found with id={cid}."

    customer = context["customer"]
    lines = [
        f"Customer #{customer['id']}: {customer['name']}",
        f"  Email:      {customer['email']}",
        f"  Country:    {customer['country']}",
        f"  Segment:    {customer['segment']}",
        f"  Created:    {customer['created_at']}",
        "",
        f"Order count:  {context['order_count']}",
        f"Total spent:  {context['total_spent']}",
        "",
        "Recent orders:",
    ]
    if context["recent_orders"]:
        for order in context["recent_orders"]:
            lines.append(
                f"  - Order #{order['id']} | {order['order_date']} | "
                f"{order['status']} | {order['total_amount']}"
            )
    else:
        lines.append("  (no orders yet)")

    return "\n".join(lines)


# ---------------------------------------------------------------------------
# PROMPT — analytics_prompt
# ---------------------------------------------------------------------------

@mcp.prompt()
def analytics_prompt(question: str) -> str:
    """Generate a structured instruction for an AI agent to answer a
    business analytics question using this server's tools and resources.

    Args:
        question: The business analytics question to answer, e.g.
            "Which customer segment generates the most revenue?"
    """
    return f"""\
You are a business analyst answering a question using the Database MCP
Server. Follow this process:

QUESTION: {question}

1. Inspect the schema.
   Read the 'db://schema' resource to understand the available tables,
   columns, and relationships before doing anything else.

2. Identify the relevant tools.
   Choose from: query_customers, customer_order_summary, top_products,
   sales_summary, order_status_summary. Do not assume a tool exists if
   it is not in this list — there is no arbitrary SQL tool available.

3. Retrieve structured data.
   Call the appropriate tool(s) with valid, well-formed parameters.
   Use multiple tool calls if the question requires combining data
   (e.g. per-customer detail plus an overall summary).

4. Avoid unsupported assumptions.
   Base your answer only on the data actually returned by the tools.
   Do not invent figures, trends, or customer details that were not
   present in the tool output.

5. Explain the result.
   Present a clear, concise answer to the question, citing the specific
   numbers returned by the tools.

6. Mention limitations when appropriate.
   Note if the available data is too limited to fully answer the
   question (e.g. small sample size, missing date range, no tool that
   directly answers a segment-level revenue breakdown).
"""


# ---------------------------------------------------------------------------
# Entrypoint
# ---------------------------------------------------------------------------

if __name__ == "__main__":
    # Ensure the database exists before serving requests.
    if not Path(db.DB_PATH).exists():
        import seed_database

        seed_database.seed()

    mcp.run()


Writing server.py


In [6]:
%%writefile client_demo.py

"""
client_demo.py

A standalone demo of an MCP client talking to the Database MCP Server.

This uses the current MCP Python SDK v2 `Client`, connected in-process
directly to the `MCPServer` instance defined in server.py (no subprocess,
no stdio transport needed — this keeps the demo simple and Colab-friendly).

Run it with:
    python client_demo.py

No API key or external service is required.
"""

from __future__ import annotations

import asyncio
import json

import seed_database
import server
from mcp import Client


def _print_header(title: str) -> None:
    print("\n" + "=" * 70)
    print(title)
    print("=" * 70)


def _print_json(data) -> None:
    print(json.dumps(data, indent=2, default=str))


async def main() -> None:
    # Make sure the demo database exists and has deterministic data.
    seed_database.seed()

    async with Client(server.mcp) as client:
        # 1. Tool discovery ---------------------------------------------
        _print_header("1. TOOL DISCOVERY")
        tools = await client.list_tools()
        for tool in tools.tools:
            print(f"  - {tool.name}: {tool.description.splitlines()[0]}")

        # 2. Read schema resource -----------------------------------------
        _print_header("2. SCHEMA RESOURCE (db://schema)")
        schema = await client.read_resource("db://schema")
        print(schema.contents[0].text)

        # 3. Search Premium customers -------------------------------------
        _print_header("3. QUERY CUSTOMERS (segment=Premium)")
        result = await client.call_tool("query_customers", {"segment": "Premium", "limit": 5})
        _print_json(result.structured_content)

        # 4. Customer analytics --------------------------------------------
        _print_header("4. CUSTOMER ORDER SUMMARY (customer_id=1)")
        result = await client.call_tool("customer_order_summary", {"customer_id": 1})
        _print_json(result.structured_content)

        # 5. Product analytics ----------------------------------------------
        _print_header("5. TOP PRODUCTS (limit=5)")
        result = await client.call_tool("top_products", {"limit": 5})
        _print_json(result.structured_content)

        # 6. Sales analytics --------------------------------------------------
        _print_header("6. SALES SUMMARY")
        result = await client.call_tool("sales_summary", {})
        _print_json(result.structured_content)

        # 6b. Order status distribution ----------------------------------------
        _print_header("6b. ORDER STATUS SUMMARY")
        result = await client.call_tool("order_status_summary", {})
        _print_json(result.structured_content)

        # 7. Dynamic customer resource ------------------------------------------
        _print_header("7. DYNAMIC CUSTOMER RESOURCE (db://customer/1)")
        result = await client.read_resource("db://customer/1")
        print(result.contents[0].text)

        # 8. Analytics prompt -----------------------------------------------------
        _print_header("8. ANALYTICS PROMPT")
        prompt_result = await client.get_prompt(
            "analytics_prompt",
            {"question": "Which customer segment generates the most revenue?"},
        )
        print(prompt_result.messages[0].content.text)

    _print_header("DEMO COMPLETE")
    print("No API key or external AI service was used. All data came from SQLite.")


if __name__ == "__main__":
    asyncio.run(main())


Writing client_demo.py


In [7]:
%%writefile tests/test_server.py

"""
tests/test_server.py

Deterministic pytest suite for the Database MCP Server.

All tests run against a temporary, isolated SQLite database (created in
a pytest tmp_path) so they never touch or corrupt the main demo database
(ecommerce.db). No API keys or external services are used.

Note on async: the project intentionally depends on nothing beyond
`mcp[cli]` and `pytest` (see requirements.txt), so these tests avoid the
`pytest-asyncio` plugin and instead drive the async MCP client with a
small `run()` helper built on `asyncio.run`.

Run with:
    pytest -q
"""

from __future__ import annotations

import asyncio
import sys
from pathlib import Path
from typing import Any, Coroutine

import pytest

# Make the project root importable when pytest is run from the repo root.
sys.path.insert(0, str(Path(__file__).resolve().parent.parent))

import database as db  # noqa: E402
import seed_database  # noqa: E402
import server  # noqa: E402
from mcp import Client  # noqa: E402


def run(coro: Coroutine[Any, Any, Any]) -> Any:
    """Run an async coroutine from a synchronous pytest test function."""
    return asyncio.run(coro)


# ---------------------------------------------------------------------------
# Fixtures
# ---------------------------------------------------------------------------

@pytest.fixture()
def test_db_path(tmp_path, monkeypatch):
    """Point both database.py and server.py at an isolated, seeded
    temporary database for the duration of a test."""
    path = tmp_path / "test_ecommerce.db"
    monkeypatch.setattr(db, "DB_PATH", path)
    monkeypatch.setattr(server.db, "DB_PATH", path)
    seed_database.seed(path)
    return path


async def _call_tool(name: str, arguments: dict[str, Any] | None = None):
    async with Client(server.mcp) as client:
        return await client.call_tool(name, arguments or {})


async def _read_resource(uri: str):
    async with Client(server.mcp) as client:
        return await client.read_resource(uri)


async def _get_prompt(name: str, arguments: dict[str, str]):
    async with Client(server.mcp) as client:
        return await client.get_prompt(name, arguments)


async def _list_tools():
    async with Client(server.mcp) as client:
        return await client.list_tools()


# ---------------------------------------------------------------------------
# 1. Database initialization
# ---------------------------------------------------------------------------

def test_database_initialization(test_db_path):
    counts = db.table_counts(test_db_path)
    assert counts["customers"] == 20
    assert counts["products"] == 10
    assert counts["orders"] == 40
    assert counts["order_items"] > 0


# ---------------------------------------------------------------------------
# 2. Tool discovery
# ---------------------------------------------------------------------------

def test_tool_discovery(test_db_path):
    tools = run(_list_tools())
    names = {t.name for t in tools.tools}
    assert names == {
        "query_customers",
        "customer_order_summary",
        "top_products",
        "sales_summary",
        "order_status_summary",
    }


# ---------------------------------------------------------------------------
# 3. Customer search
# ---------------------------------------------------------------------------

def test_customer_search_basic(test_db_path):
    result = run(_call_tool("query_customers", {"limit": 5}))
    assert result.is_error is False
    data = result.structured_content
    assert data["count"] == 5
    assert len(data["customers"]) == 5


# ---------------------------------------------------------------------------
# 4. Customer filtering
# ---------------------------------------------------------------------------

def test_customer_filtering_by_segment(test_db_path):
    result = run(_call_tool("query_customers", {"segment": "Premium", "limit": 20}))
    data = result.structured_content
    assert data["count"] > 0
    assert all(c["segment"] == "Premium" for c in data["customers"])


def test_customer_filtering_by_country(test_db_path):
    result = run(_call_tool("query_customers", {"country": "USA", "limit": 20}))
    data = result.structured_content
    assert all(c["country"] == "USA" for c in data["customers"])


# ---------------------------------------------------------------------------
# 5. Customer order summary
# ---------------------------------------------------------------------------

def test_customer_order_summary(test_db_path):
    result = run(_call_tool("customer_order_summary", {"customer_id": 1}))
    assert result.is_error is False
    data = result.structured_content
    assert data["customer"]["id"] == 1
    assert data["order_count"] >= 1
    assert "status_breakdown" in data


# ---------------------------------------------------------------------------
# 6. Top products
# ---------------------------------------------------------------------------

def test_top_products(test_db_path):
    result = run(_call_tool("top_products", {"limit": 3}))
    assert result.is_error is False
    data = result.structured_content
    assert data["count"] == 3
    revenues = [p["revenue"] for p in data["products"]]
    assert revenues == sorted(revenues, reverse=True)


# ---------------------------------------------------------------------------
# 7. Sales summary
# ---------------------------------------------------------------------------

def test_sales_summary(test_db_path):
    result = run(_call_tool("sales_summary", {}))
    assert result.is_error is False
    data = result.structured_content
    assert data["total_orders"] == 40
    assert data["unique_customers"] > 0
    assert data["total_revenue"] > 0


# ---------------------------------------------------------------------------
# 8. Order status summary
# ---------------------------------------------------------------------------

def test_order_status_summary(test_db_path):
    result = run(_call_tool("order_status_summary", {}))
    assert result.is_error is False
    data = result.structured_content
    assert set(data.keys()) == {"pending", "processing", "shipped", "delivered", "cancelled"}
    assert sum(data.values()) == 40


# ---------------------------------------------------------------------------
# 9. Schema resource
# ---------------------------------------------------------------------------

def test_schema_resource(test_db_path):
    result = run(_read_resource("db://schema"))
    text = result.contents[0].text
    assert "customers" in text
    assert "orders" in text
    assert "order_items" in text


# ---------------------------------------------------------------------------
# 10. Dynamic customer resource
# ---------------------------------------------------------------------------

def test_dynamic_customer_resource(test_db_path):
    result = run(_read_resource("db://customer/1"))
    text = result.contents[0].text
    assert "Customer #1" in text
    assert "Recent orders" in text


# ---------------------------------------------------------------------------
# 11. Analytics prompt
# ---------------------------------------------------------------------------

def test_analytics_prompt(test_db_path):
    result = run(
        _get_prompt("analytics_prompt", {"question": "Which segment spends the most?"})
    )
    text = result.messages[0].content.text
    assert "Which segment spends the most?" in text
    assert "db://schema" in text


# ---------------------------------------------------------------------------
# 12. Invalid customer ID
# ---------------------------------------------------------------------------

def test_invalid_customer_id(test_db_path):
    result = run(_call_tool("customer_order_summary", {"customer_id": -1}))
    assert result.is_error is True
    assert "Invalid customer_id" in result.content[0].text


# ---------------------------------------------------------------------------
# 13. Invalid limit
# ---------------------------------------------------------------------------

def test_invalid_limit(test_db_path):
    result = run(_call_tool("query_customers", {"limit": 999}))
    assert result.is_error is True
    assert "Invalid limit" in result.content[0].text


# ---------------------------------------------------------------------------
# 14. Invalid date
# ---------------------------------------------------------------------------

def test_invalid_date(test_db_path):
    result = run(_call_tool("sales_summary", {"start_date": "13/40/2023"}))
    assert result.is_error is True
    assert "Invalid start_date" in result.content[0].text


# ---------------------------------------------------------------------------
# 15. Nonexistent customer
# ---------------------------------------------------------------------------

def test_nonexistent_customer_tool(test_db_path):
    result = run(_call_tool("customer_order_summary", {"customer_id": 99999}))
    assert result.is_error is True
    assert "Customer not found" in result.content[0].text


def test_nonexistent_customer_resource(test_db_path):
    result = run(_read_resource("db://customer/99999"))
    text = result.contents[0].text
    assert "No customer found" in text


# ---------------------------------------------------------------------------
# Extra: invalid segment is rejected up front
# ---------------------------------------------------------------------------

def test_invalid_segment_rejected(test_db_path):
    result = run(_call_tool("query_customers", {"segment": "NotASegment"}))
    assert result.is_error is True
    assert "Invalid segment" in result.content[0].text


Writing tests/test_server.py


In [8]:
%%writefile requirements.txt

mcp[cli]
pytest

Writing requirements.txt


In [9]:
!find . -maxdepth 3 -type f | sort

./client_demo.py
./database.py
./requirements.txt
./seed_database.py
./server.py
./tests/test_server.py


In [10]:
!python seed_database.py

Database initialized successfully.

Customers:    20
Products:     10
Orders:       40
Order Items:  58

Database file: /content/database-mcp-server/ecommerce.db


In [11]:
!find . -maxdepth 2 -type f | sort

./client_demo.py
./database.py
./ecommerce.db
./__pycache__/database.cpython-313.pyc
./requirements.txt
./seed_database.py
./server.py
./tests/test_server.py


In [12]:
!pytest -q

..................                                                       [100%]
18 passed in 4.51s


In [13]:
!python client_demo.py


1. TOOL DISCOVERY
  - query_customers: Search customers using safe, optional filters.
  - customer_order_summary: Return order analytics for a single customer.
  - top_products: Identify the highest-revenue products, optionally within a category.
  - sales_summary: Provide overall business analytics, optionally scoped to a date range.
  - order_status_summary: Show the distribution of orders across all statuses.

2. SCHEMA RESOURCE (db://schema)
DATABASE SCHEMA — E-Commerce Analytics

customers
---------
  id          INTEGER  primary key
  name        TEXT     full name of the customer
  email       TEXT     unique contact email
  country     TEXT     customer's country
  segment     TEXT     one of: Standard, Premium, Enterprise
  created_at  TEXT     ISO date the customer record was created

products
--------
  id       INTEGER  primary key
  name     TEXT     product name
  category TEXT     product category (e.g. Electronics, Furniture)
  price    REAL     unit list price
  stock

In [14]:
!mcp dev server.py

⠙⠹⠸⠼⠴⠦⠧Need to install the following packages:
@modelcontextprotocol/inspector@0.15.0
Ok to proceed? (y) y

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼npm warn deprecated @modelcontextprotocol/inspector-server@0.15.0: v1 is deprecated. Upgrade to v2: npm i @modelcontextprotocol/inspector@latest. v1 gets security fixes only, published under the v1-latest tag.
⠼⠴npm warn deprecated @modelcontextprotocol/inspector-cli@0.15.0: v1 is deprecated. Upgrade to v2: npm i @modelcontextprotocol/inspector@latest. v1 gets security fixes only, published under the v1-latest tag.
⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦npm warn deprecated @modelcontextprotocol/inspector-client@0.15.0: v1 is deprecated. Upgrade to v2: npm i @modelcontextprotocol/inspector@latest. v1 gets security fixes only, published under the v1-latest tag.
⠦⠧⠇npm warn depre